# 13.7 — Sound event detection

Sound event detection labels what happened and when, often with multiple events active at the same time. This notebook implements independent sigmoid event detectors, frame grouping, smoothing, and F1 on synthetic NumPy audio. Save a copy to Drive to edit.

In [ ]:

import math
import numpy as np
import matplotlib.pyplot as plt

SEED = 1305
rng = np.random.default_rng(SEED)
FS = 8000


def sine(freq, seconds=0.35, fs=FS, phase=0.0, amp=1.0):
    n = int(seconds * fs)
    t = np.arange(n) / fs
    return amp * np.sin(2 * np.pi * freq * t + phase)


def mix_tones(freqs, seconds=0.35, fs=FS, amps=None):
    if amps is None:
        amps = np.ones(len(freqs))
    x = np.zeros(int(seconds * fs))
    for freq, amp in zip(freqs, amps):
        x = x + sine(freq, seconds, fs, amp=amp)
    scale = np.max(np.abs(x)) + 1e-12
    return x / scale


def chirp(start, stop, seconds=0.6, fs=FS):
    n = int(seconds * fs)
    t = np.arange(n) / fs
    k = (stop - start) / max(seconds, 1e-12)
    phase = 2 * np.pi * (start * t + 0.5 * k * t * t)
    return np.sin(phase)


def add_noise(x, scale, local_rng=rng):
    return x + local_rng.normal(0.0, scale, size=len(x))


def stft_mag(x, n_fft=256, hop=96):
    if len(x) < n_fft:
        x = np.pad(x, (0, n_fft - len(x)))
    frames = []
    window = np.hanning(n_fft)
    for start in range(0, len(x) - n_fft + 1, hop):
        frame = x[start:start + n_fft]
        spec = np.abs(np.fft.rfft(frame * window))
        frames.append(spec)
    if not frames:
        frames.append(np.abs(np.fft.rfft(x[:n_fft] * window)))
    return np.array(frames)


def frame_audio(x, frame_size=256, hop=128):
    frames = []
    for start in range(0, len(x) - frame_size + 1, hop):
        frames.append(x[start:start + frame_size])
    if not frames:
        frames.append(np.pad(x, (0, max(0, frame_size - len(x))))[:frame_size])
    return np.array(frames)


def normalize_rows(values):
    norms = np.linalg.norm(values, axis=1, keepdims=True)
    return values / np.maximum(norms, 1e-12)


def cosine(a, b):
    denom = np.linalg.norm(a) * np.linalg.norm(b)
    return float(np.dot(a, b) / max(denom, 1e-12))


def softmax(logits):
    shifted = logits - np.max(logits)
    exp_values = np.exp(shifted)
    return exp_values / np.sum(exp_values)


def top_frequency(x, fs=FS, n_fft=512):
    windowed = x[:n_fft]
    if len(windowed) < n_fft:
        windowed = np.pad(windowed, (0, n_fft - len(windowed)))
    mag = np.abs(np.fft.rfft(windowed * np.hanning(n_fft)))
    freqs = np.fft.rfftfreq(n_fft, 1 / fs)
    return float(freqs[int(np.argmax(mag))])


def ladder_summary(rungs):
    for rung in rungs:
        x = rung["audio"]
        label = rung.get("label", rung.get("target", "synthetic"))
        print(rung["name"], x.shape, label)



def sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))


def binary_cross_entropy(probs, labels):
    probs = np.clip(probs, 1e-7, 1 - 1e-7)
    loss = -labels * np.log(probs) - (1 - labels) * np.log(1 - probs)
    return float(np.mean(loss))


def detect_events(frame_features, weights=None, bias=None, threshold=0.5):
    if weights is None:
        weights = np.eye(frame_features.shape[1])
    if bias is None:
        bias = np.zeros(weights.shape[1])
    logits = frame_features @ weights + bias
    probs = sigmoid(logits)
    labels = probs >= threshold
    return probs, labels.astype(int)


def f1_score(y_true, y_pred):
    tp = int(np.sum((y_true == 1) & (y_pred == 1)))
    fp = int(np.sum((y_true == 0) & (y_pred == 1)))
    fn = int(np.sum((y_true == 1) & (y_pred == 0)))
    precision = tp / max(tp + fp, 1)
    recall = tp / max(tp + fn, 1)
    f1 = 2 * precision * recall / max(precision + recall, 1e-12)
    return float(f1), precision, recall


def smooth_binary(labels, min_len=2):
    smoothed = labels.copy()
    for col in range(labels.shape[1]):
        run_start = None
        for idx, value in enumerate(np.r_[labels[:, col], 0]):
            if value and run_start is None:
                run_start = idx
            if not value and run_start is not None:
                if idx - run_start < min_len:
                    smoothed[run_start:idx, col] = 0
                run_start = None
    return smoothed


def make_event_audio(labels):
    chunks = []
    for frame in labels:
        x = np.zeros(int(0.10 * FS))
        if frame[0] == 1:
            x = x + sine(700, seconds=0.10, amp=0.8)
        if frame[1] == 1:
            x = x + sine(1200, seconds=0.10, amp=0.8)
        chunks.append(x)
    return np.concatenate(chunks)


def event_features(audio, frame_count):
    segments = np.array_split(audio, frame_count)
    feats = []
    for segment in segments:
        low = abs(top_frequency(segment, n_fft=256) - 700)
        high = abs(top_frequency(segment, n_fft=256) - 1200)
        energy = np.log(np.mean(segment ** 2) + 1e-12)
        feats.append([2.5 - low / 160.0 + energy / 8.0, 2.5 - high / 180.0 + energy / 8.0])
    return np.array(feats)


def make_event_ladder():
    d1 = np.array([[0], [1], [1], [0], [1]])
    d1_multi = np.c_[d1, np.zeros(len(d1), dtype=int)]
    d2 = np.array([[1, 0], [1, 0], [0, 1], [0, 1], [0, 0], [1, 0]])
    d3 = d2.copy()
    d4 = np.array([[1, 0], [0, 0], [0, 1], [1, 0], [0, 1], [0, 0], [1, 0], [0, 1]])
    d5 = np.array([[1, 0], [1, 1], [0, 1], [0, 0], [1, 1], [0, 1], [1, 0], [0, 0], [1, 1], [0, 1]])
    ladders = [d1_multi, d2, d3, d4, d5]
    names = ["D1 sine event", "D2 two events", "D3 +noise", "D4 multi-segment", "D5 overlap"]
    out = []
    for idx, labels in enumerate(ladders):
        audio = make_event_audio(labels)
        if idx >= 2:
            audio = add_noise(audio, 0.08 + 0.03 * idx)
        out.append({"name": names[idx], "audio": audio, "labels": labels})
    return out


def evaluate_event_rung(rung, threshold=0.5, smoothing=True):
    features = event_features(rung["audio"], len(rung["labels"]))
    probs, pred = detect_events(features, threshold=threshold)
    if smoothing:
        pred = smooth_binary(pred, min_len=1)
    score, precision, recall = f1_score(rung["labels"], pred)
    return {"features": features, "probs": probs, "pred": pred, "f1": score, "precision": precision, "recall": recall, "spectrogram": stft_mag(rung["audio"])}


## The concept, built once (D1): independent sigmoid labels

For event $c$ at frame $t$, binary cross-entropy is $\mathcal{L}_{t,c}=-y_{t,c}\log p_{t,c}-(1-y_{t,c})\log(1-p_{t,c})$. The lesson probabilities $[0.1,0.7,0.8,0.2,0.6]$ with labels $[0,1,1,0,1]$ average to BCE $0.284$.

In [ ]:

lesson_probs = np.array([0.1, 0.7, 0.8, 0.2, 0.6])
lesson_labels = np.array([0, 1, 1, 0, 1])
lesson_bce = binary_cross_entropy(lesson_probs, lesson_labels)
thresholded = (lesson_probs >= 0.5).astype(int)
print("BCE", round(lesson_bce, 3))
print("thresholded", thresholded.tolist())
assert round(lesson_bce, 3) == 0.284
assert thresholded.tolist() == [0, 1, 1, 0, 1]


## F1 and threshold behavior

At threshold $0.5$, the lesson prediction has true positives $3$, false positives $0$, and false negatives $0$, so precision, recall, and F1 are all $1.000$. Raising the threshold to $0.75$ keeps only the $0.8$ frame.

In [ ]:

f1, precision, recall = f1_score(lesson_labels, thresholded)
high_threshold = (lesson_probs >= 0.75).astype(int)
print("precision", round(precision, 3), "recall", round(recall, 3), "F1", round(f1, 3))
print("threshold 0.75", high_threshold.tolist())
assert round(f1, 3) == 1.000
assert high_threshold.tolist() == [0, 0, 1, 0, 0]


## The dataset ladder

F7 audio ladder inline: sine event $ightarrow$ two-tone non-overlap $ightarrow$ plus noise $ightarrow$ multi-segment events $ightarrow$ longer/noisier overlapping events.

In [ ]:

rungs = make_event_ladder()
ladder_summary(rungs)
print("labels D1", rungs[0]["labels"].tolist())


In [ ]:

results = []
for rung in rungs:
    result = evaluate_event_rung(rung, threshold=0.5)
    results.append(result)
    print(rung["name"], "F1", round(result["f1"], 3), "precision", round(result["precision"], 3), "recall", round(result["recall"], 3))
metrics = np.array([item["f1"] for item in results])


## Results visualization

The closing figure has one artifact panel per D1–D5 rung plus a metric curve over the same ladder.

In [ ]:

fig, axes = plt.subplots(2, len(rungs), figsize=(15, 5))
for idx, rung in enumerate(rungs):
    axes[0, idx].imshow(np.log1p(results[idx]["spectrogram"]).T, origin="lower", aspect="auto")
    axes[0, idx].set_title(rung["name"])
    timeline = np.vstack([rung["labels"].T, results[idx]["pred"].T])
    axes[1, idx].imshow(timeline, aspect="auto", cmap="magma", vmin=0, vmax=1)
    axes[1, idx].set_yticks([0, 1, 2, 3])
    axes[1, idx].set_yticklabels(["true A", "true B", "pred A", "pred B"])
fig.suptitle("Spectrogram panels with true and predicted event intervals")
plt.tight_layout()
plt.figure(figsize=(6, 3))
plt.plot(np.arange(1, 6), metrics, marker="o")
plt.xticks(np.arange(1, 6), ["D1", "D2", "D3", "D4", "D5"])
plt.ylim(-0.05, 1.05)
plt.ylabel("event F1")
plt.title("F1 vs complexity")
plt.show()


## Pitfall on D5: softmax cannot represent overlap

A softmax forces one event to win each frame, so overlapping sounds are missed. Independent sigmoids plus threshold calibration and duration smoothing preserve multilabel frames.

In [ ]:

d5 = rungs[-1]
independent = evaluate_event_rung(d5, threshold=0.5, smoothing=True)
softmax_probs = np.array([softmax(row) for row in independent["features"]])
softmax_pred = np.zeros_like(d5["labels"])
winners = np.argmax(softmax_probs, axis=1)
softmax_pred[np.arange(len(winners)), winners] = 1
softmax_f1, softmax_precision, softmax_recall = f1_score(d5["labels"], softmax_pred)
print("softmax overlap F1", round(softmax_f1, 3))
print("independent sigmoid F1", round(independent["f1"], 3))
print("smoothed predictions", independent["pred"].tolist())


## Evaluate it + Practice

- Metric: frame/event F1; no-skill baseline predicts no events and gets zero recall.
- Sanity check: D1 thresholded labels should match the lesson sequence.
- Ablation: use softmax instead of independent sigmoids and D5 overlap drops.
- Failure signals: flickering one-frame events, high BCE but bad F1, and threshold sensitivity.

Practice 1: sweep thresholds from $0.3$ to $0.8$ and plot F1.


Practice 2: increase the smoothing minimum duration and inspect missed short events.